## EDA - meetup.com Data from Kaggle

### Imports and Data Download

In [ ]:
# General
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import OneHotEncoder

# Data download
import kagglehub
import shutil

# Plotting 
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

# Specific to network purposes
import networkx as nx

In [ ]:
downloaded_path = kagglehub.dataset_download("stkbailey/nashville-meetup")
dest_path = "data/raw/"
os.makedirs(os.path.dirname(dest_path), exist_ok=True)

shutil.copytree(downloaded_path, dest_path, dirs_exist_ok=True)
print(f"Dataset moved to: {dest_path}")

In [ ]:
data_dir = 'data/raw/'

# The files 
files_to_check = [
    'meta-events.csv',
    'meta-groups.csv',
    'meta-members.csv',
    'rsvps.csv',
    'member-to-group-edges.csv',
    'member-edges.csv',
    'group-edges.csv'
]

for file in files_to_check:
    file_path = os.path.join(data_dir, file)
    if os.path.exists(file_path):
        print(f"========== {file} ==========")
        # Read first 3 rows
        df = pd.read_csv(file_path, nrows=3)
        print(f"COLUMNS: {df.columns.tolist()}\n")
        print(df)
        print("\n")
    else:
        print(f"========== {file} NOT FOUND ==========\n")

### Data Processing and Cleaning

- Handle missing values and duplicates

- One-hot encode categorical variables

- Normalization/scaling?

In [ ]:
# Load data (dropping the artifact 'Unnamed: 0' columns immediately)
events_df = pd.read_csv(os.path.join(data_dir, 'meta-events.csv'))
groups_df = pd.read_csv(os.path.join(data_dir, 'meta-groups.csv'))
mem_df = pd.read_csv(os.path.join(data_dir, 'meta-members.csv'), index_col='member_id') # Index col setting attribution: Stephen Bailey
rsvps_df = pd.read_csv(os.path.join(data_dir, 'rsvps.csv')).drop(columns=['Unnamed: 0'], errors='ignore')
mem_grp_edges = pd.read_csv(os.path.join(data_dir, 'member-to-group-edges.csv'))
mem_edges = pd.read_csv(os.path.join(data_dir, 'member-edges.csv')).drop(columns=['Unnamed: 0'], errors='ignore')
grp_edges = pd.read_csv(os.path.join(data_dir, 'group-edges.csv')).drop(columns=['Unnamed: 0'], errors='ignore')

# Convert event times to datetime objects
events_df['time'] = pd.to_datetime(events_df['time'], errors='coerce')

# Merge event timestamps and group IDs into the RSVP dataframe
rsvps_master = rsvps_df.merge(events_df[['event_id', 'time', 'name']], on='event_id', how='left')
rsvps_master = rsvps_master.merge(groups_df[['group_id', 'category_name', 'category_id']], on='group_id', how='left')

all_data = {"Events": events_df, "Groups": groups_df, "Members": mem_df, "RSVPs Master": rsvps_master, 
            "Member-Group Edges": mem_grp_edges, "Member Edges": mem_edges, "Group Edges": grp_edges}

for key, value in all_data.items():
    print(f"--- {key} Table ---")
    print(value.info())
    print(f"Missing Values: \n{value.isna().sum()}")
    print(f"Duplicate Values: {value.duplicated().sum()}\n")

In [ ]:
# Out of over 125K entries in rsvps_master, fewer than 500 are missing timestamps (no other cols are missing vals); drop these rows
rsvps_master = rsvps_master.dropna()

# Fewer than 200 timestamps missing from Events table, drop these are well
events_df = events_df.dropna()

# 19,664 are missing hometown - too many to drop - replace with "Not provided"
dfs = [events_df, groups_df, mem_df, rsvps_df, mem_grp_edges, mem_edges, grp_edges]
for df in dfs:
    df.fillna("Not provided", inplace=True)

In [ ]:
# Drop superfluous columns
events_df.drop(columns='name', inplace=True)
groups_df.drop(columns=['group_name', 'category_name', 'group_urlname'], inplace=True)
mem_df.drop(columns='name', inplace=True)

In [ ]:
# Check for data types
for i, df in enumerate(dfs):
    print(f"\ndf[{i}]:")
    print(df.dtypes)

In [ ]:
events_df

In [ ]:
# All categorical variables
cats = ['event_id', 'group_id', 'category_id', 'category_name', 'organizer_id', 'member_id']

# Set up one hot encoder
encoder = OneHotEncoder(sparse_output=False)

In [ ]:
# Split up one-hot encoding into sets so as to reduce load on kernel at one time

dfs_1 = [events_df, groups_df, mem_df]

for i, df in enumerate(dfs_1):
    # Strip whitespace - assign back to dfs_1[i]
    dfs_1[i] = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

    # One hot encode
    cols_to_encode = [col for col in cats if col in dfs_1[i].columns]
    if cols_to_encode:
        one_hot_encoded = encoder.fit_transform(dfs_1[i][cols_to_encode])
        one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(cols_to_encode), index=dfs_1[i].index)
        # Drop original cols and merge encoded ones back in
        dfs_1[i] = dfs_1[i].drop(columns=cols_to_encode)
        dfs_1[i] = pd.concat([dfs_1[i], one_hot_df], axis=1)

# Unpack back to named variables
events_df, groups_df, mem_df = dfs_1

In [ ]:
# Split up one-hot encoding into sets so as to reduce load on kernel at one time

dfs_2 = [rsvps_df]

for i, df in enumerate(dfs_2):
    # Strip whitespace - assign back to dfs_1[i]
    dfs_2[i] = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

    # One hot encode
    cols_to_encode = [col for col in cats if col in dfs_2[i].columns]
    if cols_to_encode:
        one_hot_encoded = encoder.fit_transform(dfs_2[i][cols_to_encode])
        one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(cols_to_encode), index=dfs_2[i].index)
        # Drop original cols and merge encoded ones back in
        dfs_2[i] = dfs_2[i].drop(columns=cols_to_encode)
        dfs_2[i] = pd.concat([dfs_2[i], one_hot_df], axis=1)

# Unpack back to named variables
rsvps_df = dfs_2

In [ ]:
"""

-------CRASHING KERNEL---------

# Split up one-hot encoding into sets so as to reduce load on kernel at one time

dfs_3 = [mem_grp_edges]

for i, df in enumerate(dfs_3):
    # Strip whitespace - assign back to dfs_1[i]
    dfs_3[i] = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

    # One hot encode
    cols_to_encode = [col for col in cats if col in dfs_3[i].columns]
    if cols_to_encode:
        one_hot_encoded = encoder.fit_transform(dfs_3[i][cols_to_encode])
        one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(cols_to_encode), index=dfs_3[i].index)
        # Drop original cols and merge encoded ones back in
        dfs_3[i] = dfs_3[i].drop(columns=cols_to_encode)
        dfs_3[i] = pd.concat([dfs_3[i], one_hot_df], axis=1)

# Unpack back to named variables
mmem_grp_edges = dfs_3"""

In [ ]:
# Split up one-hot encoding into sets so as to reduce load on kernel at one time

dfs_4 = [mem_edges, grp_edges]

for i, df in enumerate(dfs_4):
    # Strip whitespace - assign back to dfs_1[i]
    dfs_4[i] = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

    # One hot encode
    cols_to_encode = [col for col in cats if col in dfs_4[i].columns]
    if cols_to_encode:
        one_hot_encoded = encoder.fit_transform(dfs_4[i][cols_to_encode])
        one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(cols_to_encode), index=dfs_4[i].index)
        # Drop original cols and merge encoded ones back in
        dfs_4[i] = dfs_4[i].drop(columns=cols_to_encode)
        dfs_4[i] = pd.concat([dfs_4[i], one_hot_df], axis=1)

# Unpack back to named variables
mem_edges, grp_edges = dfs_4

In [ ]:
rsvps_series = pd.Series(rsvps_master.groupby('event_id').size(), name="num_rsvps")
event_sizes = rsvps_series.to_frame()
event_sizes

### Plotting and Exploration

In [ ]:
plt.figure(figsize=(16, 5))

# --- Plot 1: Event Size Distribution ---
# How many RSVPs does a typical event get?

plt.subplot(1, 3, 1)
sns.histplot(event_sizes, x='num_rsvps',  bins=50, color='indigo')
plt.yscale('log') # Added to fix log scale definition issue within sns
plt.title('(a) RSVPs per Event (Log Scale)')
plt.xlabel('Number of RSVPs')
plt.ylabel('Frequency (Log)')

# --- Plot 2: Category Popularity ---
# Which domains drive the most interaction? 
plt.subplot(1, 3, 2)
category_counts = rsvps_master['category_name'].value_counts().head(10)
sns.barplot(y=category_counts.index, x=category_counts.values, palette='viridis')
plt.title('(b) Total RSVPs by Category')
plt.xlabel('RSVP Count')

# --- Plot 3: Temporal Event Frequency ---
# Identifying the timeline for your train/val/test split
plt.subplot(1, 3, 3)
rsvps_master.set_index('time').resample('ME').size().plot(color='teal', lw=2)
plt.title('(c) RSVP Volume Over Time (Monthly)')
plt.xlabel('Event Date')
plt.ylabel('RSVPs')

plt.tight_layout()
plt.show()

print("\n--- Event Size Statistics ---")
print(event_sizes.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

In [ ]:
mem_edges.head()

In [ ]:
print(mem_edges.columns.tolist())
print(mem_edges[['member1', 'member2', 'weight']].head())

In [ ]:
import networkx as nx

subset = mem_edges.head(5000).dropna(subset=['member1', 'member2']).copy()


G_sample = nx.from_pandas_edgelist(
    subset,
    source='member1',
    target='member2',
    edge_attr='weight',
    create_using=nx.Graph()
)

assert G_sample is not None, "Graph creation returned None"
print(f"Nodes: {G_sample.number_of_nodes()}, Edges: {G_sample.number_of_edges()}")

In [ ]:
print("--- Analyzing Pre-computed Member Graph ---")
print(f"Total Edges Provided: {len(mem_edges)}")

# Build a graph sample to calculate structural properties efficiently
# Using top 5,000 edges to avoid memory issues and excessive computational time during EDA
subset = mem_edges.head(5000).copy().reset_index(drop=True)
G_sample = nx.from_pandas_edgelist(
    subset,
    source='member1',
    target='member2',
    edge_attr='weight',
    create_using=nx.Graph())
# Set attributes , ex btwnness centrality
bb = nx.betweenness_centrality(G_sample)
nx.set_node_attributes(G_sample, bb, 'betweenness')

print(f"Sample Graph Nodes: {G_sample.number_of_nodes()}")
print(f"Sample Graph Edges: {G_sample.number_of_edges()}")

# Calculate Degree Distribution
degrees = [d for n, d in G_sample.degree()]

plt.figure(figsize=(10, 5))
sns.histplot(degrees, bins=50, color='coral')
plt.yscale('log')
plt.title('Member Degree Distribution (Sampled Co-occurrence Graph)')
plt.xlabel('Degree (Number of Connections)')
plt.ylabel('Frequency (Log Scale)')
plt.show()

# Check weight distribution to validate edge decay viability
print("\n--- Edge Weight Summary ---")
print(mem_edges['weight'].describe())

In [ ]:
print(f"Type of graph object: {type(G_sample)}")